In [ ]:
# [COLAB_BOOTSTRAP_v2_modular]
# Mount Drive + clone upstream moment_detr + extract features to LOCAL SSD.
# Run once at the top of any modular notebook. Idempotent within a session.
import os, sys, subprocess, time, tarfile, shutil
from pathlib import Path

# --- 1. Drive --------------------------------------------------------------
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/thumbnail_project')
REPO_DIR   = DRIVE_ROOT / 'thumbnail-consistency'
assert REPO_DIR.exists(), (
    f"Upload your repo folder to {REPO_DIR} first. "
    f"Found nothing at that path."
)

# --- 2. Put repo on sys.path (so `from src.*` works) -----------------------
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

# --- 2b. Symlink data dirs INTO the repo so relative paths work ------------
# Notebooks reference 'processed/X.jsonl', 'outputs/Y.png' etc., assuming
# those dirs live INSIDE the repo. In your Drive layout they sit ALONGSIDE
# the repo. Symlinks bridge that gap.
DATA_DIRS_TO_LINK = [
    'processed', 'outputs',
    'qvh_videos', 'qvh_frames', 'qvh_thumbnails',
    'qvh_frames_within_neg', 'qvh_frames_multi',
    'checkpoints',
]
for d in DATA_DIRS_TO_LINK:
    src = DRIVE_ROOT / d
    dst = REPO_DIR / d
    if not src.exists():
        continue
    if dst.exists() or dst.is_symlink():
        continue
    try:
        os.symlink(src, dst)
        print(f"  symlinked  {d}/  ->  {src}")
    except OSError as e:
        print(f"  WARN: couldn't symlink {d}: {e}")

# --- 3. Extract M1 features to LOCAL SSD (NOT to Drive) --------------------
# Drive cannot reliably store 19k small .npz files; we extract every session.
ARCHIVE   = DRIVE_ROOT / 'moment_detr_features.tar.gz'
LOCAL_OUT = Path('/content/features_local')


def _find_feature_dirs(root):
    clip = text = None
    for r, dirs, _ in os.walk(root):
        for d in dirs:
            p = Path(r) / d
            if d == 'clip_features' and len(list(p.glob('*.npz'))) > 1000:
                clip = p
            if d == 'clip_text_features' and len(list(p.glob('*.npz'))) > 1000:
                text = p
    return clip, text


CLIP_FEAT_DIR, TEXT_FEAT_DIR = (None, None)
if LOCAL_OUT.exists():
    CLIP_FEAT_DIR, TEXT_FEAT_DIR = _find_feature_dirs(LOCAL_OUT)

if CLIP_FEAT_DIR and TEXT_FEAT_DIR:
    print(f"[features] already extracted this session.")
else:
    assert ARCHIVE.exists(), f"Archive missing: {ARCHIVE}"
    if LOCAL_OUT.exists():
        shutil.rmtree(LOCAL_OUT)
    LOCAL_OUT.mkdir(parents=True)
    print(f"[features] extracting {ARCHIVE.stat().st_size/1e9:.2f} GB -> {LOCAL_OUT}")
    t0 = time.time()
    with tarfile.open(ARCHIVE) as tf:
        tf.extractall(LOCAL_OUT)
    print(f"[features] extracted in {time.time()-t0:.0f}s")
    CLIP_FEAT_DIR, TEXT_FEAT_DIR = _find_feature_dirs(LOCAL_OUT)

assert CLIP_FEAT_DIR and TEXT_FEAT_DIR, "Feature dirs not found after extraction."

# --- 4. Clone upstream Moment-DETR (used by src.models.moment_detr) --------
MD_DIR = Path('/content/moment_detr_upstream')
if not MD_DIR.exists():
    subprocess.run(['git', 'clone',
                    'https://github.com/jayleicn/moment_detr.git',
                    str(MD_DIR)], check=True)
if str(MD_DIR) not in sys.path:
    sys.path.insert(0, str(MD_DIR))

# --- 5. Expose path variables for downstream cells -------------------------
# Any cell that reads configs/*.yaml can override:
#   cfg['paths']['feature_dir']   = str(CLIP_FEAT_DIR)
#   cfg['paths']['text_feat_dir'] = str(TEXT_FEAT_DIR)
# (configs already point at /content/features_local/... so usually no override needed.)

# --- 6. Verification report -------------------------------------------------
import torch
print()
print(f"Repo            : {REPO_DIR}")
print(f"Device          : {'cuda' if torch.cuda.is_available() else 'cpu (WARN: slow)'}")
print(f"clip_features   : {CLIP_FEAT_DIR}  ({len(list(CLIP_FEAT_DIR.glob('*.npz'))):,} files)")
print(f"clip_text_feats : {TEXT_FEAT_DIR}  ({len(list(TEXT_FEAT_DIR.glob('*.npz'))):,} files)")
print(f"src/ importable : {(REPO_DIR / 'src' / '__init__.py').exists()}")
print()
print("Bootstrap done. You can now run the rest of the notebook.")


# Run full consistency evaluation + ablation table (Day 2)

Reads:
- `processed/consistency_eval.jsonl`
- `outputs/predictions/m2_generated.jsonl` (from train_clipcap.ipynb)

Produces:
- `outputs/predictions/consistency_scores.jsonl`
- `outputs/figures/score_distribution.png`
- `outputs/figures/ablation_table.csv`

Ablation query sources (per FINAL_DESIGN section 9 + configs/eval.yaml):
  1. **gt_query**   — GT QVH query (ceiling)
  2. **clipcap_k3** — 3 nucleus-sampled M2 queries
  3. **clipcap_k1** — single beam-3 M2 query
  4. **thumb_only** — alpha=0 (no query path)
  5. **random_query** — control

In [ ]:
import sys, json
from pathlib import Path
PROJECT = Path('.').resolve()
if str(PROJECT) not in sys.path: sys.path.insert(0, str(PROJECT))

import numpy as np, pandas as pd, torch
from tqdm.auto import tqdm
from src.utils.io import load_jsonl, save_jsonl, load_yaml
from src.models.moment_detr import MomentDETRFeaturePredictor
from src.scoring.consistency import ConsistencyScorer, ScoringConfig
from src.eval.metrics import summarize_separation

ccfg = load_yaml('configs/consistency.yaml')
ecfg = load_yaml('configs/eval.yaml')
scoring_cfg = ScoringConfig(alpha=ccfg['alpha'], tau=ccfg['tau'],
                             top_p=ccfg['top_p'],
                             chunk_window=ccfg['chunk_window'],
                             chunk_stride=ccfg['chunk_stride'])
m1 = MomentDETRFeaturePredictor(
    ckpt_path=ccfg['paths']['m1_ckpt'],
    repo_root=ccfg['paths']['repo_root'],
)
scorer = ConsistencyScorer(m1, ccfg['paths']['feature_dir'], scoring_cfg)

In [ ]:
pairs = load_jsonl(ecfg['consistency_eval_jsonl'])
gen   = load_jsonl('outputs/predictions/m2_generated.jsonl')
gen_by_pid = {g['pair_id']: g['queries'] for g in gen}
print(f'pairs={len(pairs)}  generations={len(gen_by_pid)}')

import random
random.seed(2026)
all_queries_for_random = [p['thumb_query'] for p in pairs]

def queries_for(pair, source):
    if source == 'gt_query':     return [pair['thumb_query']]
    if source == 'clipcap_k3':   return gen_by_pid.get(pair['pair_id'], [pair['thumb_query']])
    if source == 'clipcap_k1':   return [gen_by_pid.get(pair['pair_id'], [pair['thumb_query']])[0]]
    if source == 'thumb_only':   return [pair['thumb_query']]  # ignored via alpha=0
    if source == 'random_query': return [random.choice(all_queries_for_random)]
    raise ValueError(source)

def alpha_for(source):
    return 0.0 if source == 'thumb_only' else scoring_cfg.alpha

In [ ]:
out_rows = []
for source in ecfg['ablation_query_sources']:
    scorer.cfg.alpha = alpha_for(source)
    rows = []
    for p in tqdm(pairs, desc=source):
        out = scorer.score(
            thumbnail_path=p['frame_path'],
            vid=p['video_vid'],
            queries=queries_for(p, source),
        )
        rows.append({'pair_id': p['pair_id'], 'label': p['label'],
                     'source': source, 'score': out['C']})
    save_jsonl(rows, f"outputs/predictions/scores_{source}.jsonl")
    scores = [r['score'] for r in rows]
    labels = [r['label'] for r in rows]
    summary = summarize_separation(scores, labels)
    summary['source'] = source
    out_rows.append(summary)
    print(source, '->', {k: round(v, 3) if isinstance(v, float) else v
                          for k, v in summary.items()})
scorer.cfg.alpha = ccfg['alpha']  # restore

In [ ]:
df = pd.DataFrame(out_rows).set_index('source')
Path('outputs/figures').mkdir(parents=True, exist_ok=True)
df.to_csv('outputs/figures/ablation_table.csv')
df

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, len(ecfg['ablation_query_sources']),
                          figsize=(4 * len(ecfg['ablation_query_sources']), 4),
                          sharey=True)
for ax, src in zip(axes, ecfg['ablation_query_sources']):
    rows = load_jsonl(f'outputs/predictions/scores_{src}.jsonl')
    pos = [r['score'] for r in rows if r['label'] == 1]
    neg = [r['score'] for r in rows if r['label'] == 0]
    ax.hist(pos, bins=25, alpha=0.5, label='pos')
    ax.hist(neg, bins=25, alpha=0.5, label='neg')
    auc = df.loc[src, 'auc']
    ax.set_title(f"{src}\nAUC={auc:.3f}")
    ax.legend()
plt.tight_layout()
plt.savefig('outputs/figures/score_distribution.png', dpi=150, bbox_inches='tight')
plt.show()